## Calculating Settlements From Trades

In many cases, we use the settlements provided by the exchange, but understanding the source of the
settlement price in different markets is useful for at least a couple of reasons:
1. it helps understand the small but important differences among different markets,
2. you want to make markets in TAS contracts, which we will try to cover by the end of the course.

This assignment is a chance to start building your understanding and intuition about futures markets
along the lines of 1.


You may find the following code a useful template from `notebook/calendars.ipynb`.

In [1]:
import math
from zoneinfo import ZoneInfo

import databento as db
import numpy as np
import pandas as pd

from finm37000 import (
    as_ct,
    get_cme_session_end,
    get_databento_api_key,
    get_official_stats,
    temp_env,
)

tz_chicago = ZoneInfo("America/Chicago")
cme = "GLBX.MDP3"

with temp_env(DATABENTO_API_KEY=get_databento_api_key()):
    client = db.Historical()

In [2]:
def calc_vwap(price, volume):
    return (price * volume).sum() / volume.sum()


def get_official_settle(stat_df, dt, symbol):
    return stat_df[
        (stat_df.index.get_level_values(0) == dt.date())
        & (stat_df.index.get_level_values(1) == symbol)
    ]["Settlement price"].iloc[0]


def round_to_tick(x, min_price_change):
    return round(x / min_price_change) * min_price_change

In [3]:
date = pd.Timestamp("2025-10-09", tz=tz_chicago)

In [4]:
product = "CL"
months = ("X5", "Z5", "F6")
settle_start = date + pd.Timedelta(hours=13, minutes=28)
settle_end = date + pd.Timedelta(hours=13, minutes=30)
tick_size = 0.01

legs = [f"{product}{month}" for month in months]
spreads = {
    legs[0]: [],
    legs[1]: [
        f"{legs[0]}-{legs[1]}",
    ],
    legs[2]: [f"{legs[0]}-{legs[2]}", f"{legs[1]}-{legs[2]}"],
}
leg2_spread_month_divisors = (2, 1)
vwap_symbols = sum(spreads.values(), [])
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols=vwap_symbols + legs,
    start=settle_start,
    end=settle_end,
)
trade_snippet = trade_snippet_raw.to_df()
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trades = trade_snippet.set_index(["local_time", "sequence"])
session_end = get_cme_session_end(date)
next_session_end = get_cme_session_end(date + pd.Timedelta(days=1))

raw_stats = client.timeseries.get_range(
    dataset=cme,
    schema="statistics",
    symbols=legs,
    start=session_end,
    end=next_session_end,
)
instrument_defs = client.timeseries.get_range(
    dataset=cme,
    schema="definition",
    symbols=legs,
    start=date.date(),
)
stats = get_official_stats(raw_stats.to_df(), instrument_defs.to_df())

settle_vwap = {
    contract: round_to_tick(calc_vwap(df["price"], df["size"]), tick_size)
    for contract, df in trades.groupby("symbol")
}
settle_size = {contract: df["size"].sum() for contract, df in trades.groupby("symbol")}

calculated_settles = {legs[0]: settle_vwap[legs[0]]}
first_official_settle = get_official_settle(stats, date, legs[0])
assert math.isclose(
    calculated_settles[legs[0]],
    first_official_settle,
    abs_tol=tick_size / 2,
)

calculated_settles[legs[1]] = settle_vwap[legs[0]] - settle_vwap[spreads[legs[1]][0]]
second_official_settle = get_official_settle(stats, date, legs[1])
assert math.isclose(
    calculated_settles[legs[1]],
    second_official_settle,
    abs_tol=tick_size / 2,
)

intermediate_settles = [
    calculated_settles[legs[i]] - settle_vwap[spreads[legs[2]][i]] for i in range(2)
]
intermediate_weights = [
    settle_size[spreads[legs[2]][i]] / leg2_spread_month_divisors[i] for i in range(2)
]
calculated_settles[legs[2]] = calc_vwap(
    np.array(intermediate_settles),
    np.array(intermediate_weights),
)
third_official_settle = get_official_settle(stats, date, legs[2])

assert math.isclose(
    calculated_settles[legs[2]],
    third_official_settle,
    abs_tol=tick_size / 2.0,
)

1. For the `ZSX5`, `ZSF6`, and `ZSH6` contracts on 2025-10-09, calculate their settlement prices from the trade data
   to match the official settlements from the exchange. Use `assert` to compare the official settlements
   against your calculation.

You need to identify the correct settlement time, tick size, and month divisors.
It is possible for the asserts to pass with the wrong spread month divisors.
Did you use the right ones?

In [5]:
product = "ZS"
months = ("X5", "F6", "H6")
settle_start = date + pd.Timedelta(hours=13, minutes=14)
settle_end = date + pd.Timedelta(hours=13, minutes=15)
tick_size = 0.25

legs = [f"{product}{month}" for month in months]
spreads = {
    legs[0]: [],
    legs[1]: [
        f"{legs[0]}-{legs[1]}",
    ],
    legs[2]: [f"{legs[0]}-{legs[2]}", f"{legs[1]}-{legs[2]}"],
}
leg2_spread_month_divisors = (1, 3)
vwap_symbols = sum(spreads.values(), [])
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols=vwap_symbols + legs,
    start=settle_start,
    end=settle_end,
)
trade_snippet = trade_snippet_raw.to_df()
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trades = trade_snippet.set_index(["local_time", "sequence"])

settle_vwap = {
    contract: round_to_tick(calc_vwap(df["price"], df["size"]), tick_size)
    for contract, df in trades.groupby("symbol")
}
settle_size = {contract: df["size"].sum() for contract, df in trades.groupby("symbol")}
raw_stats = client.timeseries.get_range(
    dataset=cme,
    schema="statistics",
    symbols=legs,
    start=session_end,
    end=next_session_end,
)
instrument_defs = client.timeseries.get_range(
    dataset=cme,
    schema="definition",
    symbols=legs,
    start=date.date(),
)
stats = get_official_stats(raw_stats.to_df(), instrument_defs.to_df())

calculated_settles = {legs[0]: settle_vwap[legs[0]]}
first_official_settle = get_official_settle(stats, date, legs[0])
assert math.isclose(
    calculated_settles[legs[0]],
    first_official_settle,
    abs_tol=tick_size / 2,
)
calculated_settles[legs[1]] = settle_vwap[legs[0]] - settle_vwap[spreads[legs[1]][0]]
second_official_settle = get_official_settle(stats, date, legs[1])

assert math.isclose(
    calculated_settles[legs[1]],
    second_official_settle,
    abs_tol=tick_size / 2,
)

intermediate_settles = [
    calculated_settles[legs[i]] - settle_vwap[spreads[legs[2]][i]] for i in range(2)
]
intermediate_weights = [
    settle_size[spreads[legs[2]][i]] / leg2_spread_month_divisors[i] for i in range(2)
]
calculated_settles[legs[2]] = calc_vwap(
    np.array(intermediate_settles),
    np.array(intermediate_weights),
)
third_official_settle = get_official_settle(stats, date, legs[2])
assert math.isclose(
    calculated_settles[legs[2]],
    third_official_settle,
    abs_tol=tick_size / 2.0,
)

2. For the `ESZ5` and `ESH6` contracts on 2025-10-09, calculate their settlement prices from the trade data
   to match the official settlements from the exchange. Use `assert` to compare the official settlements
   against your calculation.

(N.B., the trade volumes drop off with increasing
days to expiration in ES making it necessary to use alternate procedures to settle. Feel free to look
at the CME documentation: https://cmegroupclientsite.atlassian.net/wiki/spaces/EPICSANDBOX/pages/457418067/E-Mini+Standard+and+Poors+500+Futures)

You need to identify the correct settlement window and tick size for settlement and realize
that equity index spreads represent the opposite spread as the standard spreads, like CL and ZS,
thus a subtraction above turns into an addition here.

In [6]:
product = "ES"
months = ("Z5", "H6")
settle_start = date + pd.Timedelta(hours=14, minutes=59, seconds=30)
settle_end = date + pd.Timedelta(hours=15)
tick_size = 0.25

legs = [f"{product}{month}" for month in months]
spreads = {
    legs[0]: [],
    legs[1]: [
        f"{legs[0]}-{legs[1]}",
    ],
}
vwap_symbols = sum(spreads.values(), [])
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols=vwap_symbols + legs,
    start=settle_start,
    end=settle_end,
)
trade_snippet = trade_snippet_raw.to_df()
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trades = trade_snippet.set_index(["local_time", "sequence"])

settle_vwap = {
    contract: round_to_tick(calc_vwap(df["price"], df["size"]), tick_size)
    for contract, df in trades.groupby("symbol")
}
settle_size = {contract: df["size"].sum() for contract, df in trades.groupby("symbol")}
raw_stats = client.timeseries.get_range(
    dataset=cme,
    schema="statistics",
    symbols=legs,
    start=session_end,
    end=next_session_end,
)
instrument_defs = client.timeseries.get_range(
    dataset=cme,
    schema="definition",
    symbols=legs,
    start=date.date(),
)
stats = get_official_stats(raw_stats.to_df(), instrument_defs.to_df())

calculated_settles = {legs[0]: settle_vwap[legs[0]]}
first_official_settle = get_official_settle(stats, date, legs[0])
assert math.isclose(
    calculated_settles[legs[0]],
    first_official_settle,
    abs_tol=tick_size / 2,
)

calculated_settles[legs[1]] = settle_vwap[legs[0]] + settle_vwap[spreads[legs[1]][0]]
second_official_settle = get_official_settle(stats, date, legs[1])
assert math.isclose(
    calculated_settles[legs[1]],
    second_official_settle,
    abs_tol=tick_size / 2,
)

3. For the `6EZ5` contracts on 2025-10-09, calculate their settlement prices from the trade data
   to match the official settlements from the exchange. Use `assert` to compare the official settlements
   against your calculation.

Figure out the correct settlement window and tick size.

In [7]:
product = "6E"
months = ("Z5",)
settle_start = date + pd.Timedelta(hours=13, minutes=59, seconds=30)
settle_end = date + pd.Timedelta(hours=14)
tick_size = 0.00005

legs = [f"{product}{month}" for month in months]
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols=legs,
    start=settle_start,
    end=settle_end,
)
trade_snippet = trade_snippet_raw.to_df()
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trades = trade_snippet.set_index(["local_time", "sequence"])

settle_vwap = {
    contract: round_to_tick(calc_vwap(df["price"], df["size"]), tick_size)
    for contract, df in trades.groupby("symbol")
}
settle_size = {contract: df["size"].sum() for contract, df in trades.groupby("symbol")}
raw_stats = client.timeseries.get_range(
    dataset=cme,
    schema="statistics",
    symbols=legs,
    start=session_end,
    end=next_session_end,
)
instrument_defs = client.timeseries.get_range(
    dataset=cme,
    schema="definition",
    symbols=legs,
    start=date.date(),
)
stats = get_official_stats(raw_stats.to_df(), instrument_defs.to_df())

calculated_settles = {legs[0]: settle_vwap[legs[0]]}
first_official_settle = get_official_settle(stats, date, legs[0])
assert math.isclose(
    calculated_settles[legs[0]],
    first_official_settle,
    abs_tol=tick_size / 2,
)

4. Check the volume of trading in `6EX5`, `6EH5`, and their spread contracts against `6EZ5` in the five minutes
   before the settlement window. You should be able to tell why the settlement procedures are different
   for these contracts from the preceding ones. You may be interested to read about their settlement
   procedures on the CME website, but you do not need to know those details for this course.

In [8]:
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols="6E.FUT",
    stype_in="parent",
    start=settle_start - pd.Timedelta(minutes=5),
    end=settle_end,
)

 You did not need to notice for the assignment that these spreads are quoted
in yet another convention.

In [9]:
trade_snippet_raw.to_df().groupby("symbol").sum("size")["size"]

symbol
6EH6            1
6EH6-6EZ5      27
6EX5            4
6EZ5         1322
6EZ5-6EX5       1
Name: size, dtype: uint32

There are 27 trades in `6EH6-6EZ5` and 1 trade in `6EZ5-6EX5` for a total of 28 total spread trades in that window.
There is not enough active trading in the spreads to use as part of the settlement procedure. Do you think there
would be more spread trading on some days?

5. As discussed in class, not all trades appear in the electronic trade log, and that can affect the calculated settlement. Apply the daily settlement calculation for the May 2020 Crude oil contract CLK0 on 2020-04-20. Try to check it against the official settlement (there were data issues around this date, and you will not lose points if you cannot find the official settlement).

In [26]:
date = pd.Timestamp("2020-04-20", tz=tz_chicago)
session_end = get_cme_session_end(date)
next_session_end = get_cme_session_end(date + pd.Timedelta(days=1))
product = "CL"
months = ("K0",)
settle_start = date + pd.Timedelta(hours=13, minutes=28)
settle_end = date + pd.Timedelta(hours=13, minutes=30)
tick_size = 0.01

legs = [f"{product}{month}" for month in months]
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols=legs,
    start=settle_start,
    end=settle_end,
)
trade_snippet = trade_snippet_raw.to_df()
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trades = trade_snippet.set_index(["local_time", "sequence"])

settle_vwap = {
    contract: round_to_tick(calc_vwap(df["price"], df["size"]), tick_size)
    for contract, df in trades.groupby("symbol")
}
settle_size = {contract: df["size"].sum() for contract, df in trades.groupby("symbol")}
raw_stats = client.timeseries.get_range(
    dataset=cme,
    schema="statistics",
    symbols=legs,
    start=session_end,
    end=next_session_end,
)
instrument_defs = client.timeseries.get_range(
    dataset=cme,
    schema="definition",
    symbols=legs,
    start=date.date(),
)
stats = get_official_stats(raw_stats.to_df(), instrument_defs.to_df())

calculated_settles = {legs[0]: settle_vwap[legs[0]]}
first_official_settle = get_official_settle(stats, date, legs[0])
calculated_settles, first_official_settle

({'CLK0': -37.75}, np.float64(nan))

The vwap from the trade data is negative, and the official settle is missing!
The official settle is mis-dated in the official data feed: it is dated `2024-04-17` (the previous trade date):

In [35]:
get_official_settle(stats, date - pd.Timedelta(days=3), legs[0])

np.float64(-37.63)

You may look at the full stats table to see that the received and event time for the reference date
matches typical `2020-04-20` data:

In [37]:
df = raw_stats.to_df()
cols = [
    "ts_event",
    "price",
    "ts_ref",
    "stat_type",
    "stat_flags",
    "symbol",
]
df[~df["ts_ref"].isna()][cols]

,ts_event,price,ts_ref,stat_type,stat_flags,symbol
ts_recv,,,,,,
2020-04-20 21:43:42.035255990+00:00,2020-04-20 21:43:42.030066527+00:00,-37.63,2020-04-17 00:00:00+00:00,3,3,CLK0
2020-04-21 01:30:07.220817236+00:00,2020-04-21 01:30:07.220696211+00:00,NaN,2020-04-20 00:00:00+00:00,9,0,CLK0
2020-04-21 01:30:07.220817236+00:00,2020-04-21 01:30:07.220696211+00:00,NaN,2020-04-20 00:00:00+00:00,6,0,CLK0
2020-04-21 13:18:48.517818958+00:00,2020-04-21 13:18:48.517719857+00:00,NaN,2020-04-20 00:00:00+00:00,9,0,CLK0
2020-04-21 13:18:48.517818958+00:00,2020-04-21 13:18:48.517719857+00:00,NaN,2020-04-20 00:00:00+00:00,6,0,CLK0
2020-04-21 18:34:07.836962209+00:00,2020-04-21 18:34:07.836769685+00:00,10.01,2020-04-21 00:00:00+00:00,3,2,CLK0
2020-04-21 18:45:52.765684721+00:00,2020-04-21 18:45:52.765459269+00:00,10.01,2020-04-21 00:00:00+00:00,3,2,CLK0


You can also confirm that the correct data from `2020-04-17` is not the negative values:

In [42]:
date = pd.Timestamp("2020-04-17", tz=tz_chicago)
session_end = get_cme_session_end(date)
next_session_end = get_cme_session_end(date + pd.Timedelta(days=2))
settle_start = date + pd.Timedelta(hours=13, minutes=28)
settle_end = date + pd.Timedelta(hours=13, minutes=30)

legs = [f"{product}{month}" for month in months]
trade_snippet_raw = client.timeseries.get_range(
    dataset=cme,
    schema="tbbo",
    symbols=legs,
    start=settle_start,
    end=settle_end,
)
trade_snippet = trade_snippet_raw.to_df()
trade_snippet["local_time"] = as_ct(trade_snippet["ts_event"])
trades = trade_snippet.set_index(["local_time", "sequence"])

settle_vwap = {
    contract: round_to_tick(calc_vwap(df["price"], df["size"]), tick_size)
    for contract, df in trades.groupby("symbol")
}
settle_size = {contract: df["size"].sum() for contract, df in trades.groupby("symbol")}
raw_stats = client.timeseries.get_range(
    dataset=cme,
    schema="statistics",
    symbols=legs,
    start=session_end,
    end=next_session_end,
)
instrument_defs = client.timeseries.get_range(
    dataset=cme,
    schema="definition",
    symbols=legs,
    start=date.date(),
)
stats = get_official_stats(raw_stats.to_df(), instrument_defs.to_df())

calculated_settles = {legs[0]: settle_vwap[legs[0]]}
first_official_settle = get_official_settle(stats, date, legs[0])
calculated_settles, first_official_settle

({'CLK0': 18.27}, np.float64(18.27))